In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/DTD-Wijesinghe/IT3091---Machine-Learning-Assignment.git"
STUDENT_ROLE = "IT24101090 - Jayakody N.D"
STUDENT_NAME = input("Enter your real GitHub display name: ").strip()
STUDENT_EMAIL = input("Enter your verified GitHub email: ").strip()
CLONE_DIR = Path("/content/IT3091---Machine-Learning-Assignment")

if not CLONE_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(CLONE_DIR)], check=True)
os.chdir(CLONE_DIR)
subprocess.run(["git", "fetch", "origin"], check=True)
subprocess.run(["git", "checkout", "main"], check=True)
subprocess.run(["git", "pull", "--ff-only", "origin", "main"], check=True)
subprocess.run(["git", "config", "user.name", STUDENT_NAME], check=True)
subprocess.run(["git", "config", "user.email", STUDENT_EMAIL], check=True)
print("Ready for", STUDENT_ROLE)
print("Authenticate your own GitHub account before running the final cell.")

# Colombo wholesale price preparation

This notebook reads the daily Colombo wholesale workbook, keeps daily observations, creates lag and rolling features, and labels each row with the following week average price.

In [ ]:
from pathlib import Path
import re
import pandas as pd

ROOT = Path.cwd()
source = ROOT / 'data' / 'raw' / '2018-2025 Daily Wholesale Prices (Colombo).xlsx'
rows = []
for sheet in pd.ExcelFile(source).sheet_names:
    raw = pd.read_excel(source, sheet_name=sheet, header=3)
    item_col = raw.columns[3]
    for _, row in raw[raw[item_col].notna()].iterrows():
        crop = re.sub(r'\s+', ' ', str(row[item_col]).strip()).title()
        for column in raw.columns[4:]:
            date = pd.to_datetime(column, errors='coerce')
            value = pd.to_numeric(row[column], errors='coerce')
            if pd.notna(date) and pd.notna(value) and value >= 0:
                rows.append((pd.Timestamp(date).normalize(), crop, float(value)))
daily = pd.DataFrame(rows, columns=['Date', 'Crop', 'Price']).groupby(['Date', 'Crop'], as_index=False)['Price'].mean()
daily = daily.sort_values(['Crop', 'Date']).reset_index(drop=True)
daily['WeekStart'] = daily['Date'] - pd.to_timedelta(daily['Date'].dt.weekday, unit='D')
weekly = daily.groupby(['Crop', 'WeekStart'], as_index=False)['Price'].mean().rename(columns={'Price': 'NextWeekWholesalePrice_RsKg'})
weekly['ForecastDate'] = weekly['WeekStart'] - pd.Timedelta(days=7)
target = weekly.rename(columns={'ForecastDate': 'NextWeekStart'})
out = daily.copy()
out['NextWeekStart'] = out['WeekStart'] + pd.Timedelta(days=7)
out = out.merge(target[['Crop', 'NextWeekStart', 'NextWeekWholesalePrice_RsKg']], on=['Crop', 'NextWeekStart'], how='inner')
grouped = out.groupby('Crop', group_keys=False)
out['CurrentPrice_RsKg'] = out['Price']
out['Lag1Price_RsKg'] = grouped['Price'].shift(1)
out['Lag2Price_RsKg'] = grouped['Price'].shift(2)
out['Lag3Price_RsKg'] = grouped['Price'].shift(3)
out['Rolling7Mean_RsKg'] = grouped['Price'].transform(lambda s: s.rolling(7, min_periods=3).mean())
out['Rolling28Mean_RsKg'] = grouped['Price'].transform(lambda s: s.rolling(28, min_periods=7).mean())
out['Rolling7Std_RsKg'] = grouped['Price'].transform(lambda s: s.rolling(7, min_periods=3).std())
out['Month'] = out['Date'].dt.month
out['WeekOfYear'] = out['Date'].dt.isocalendar().week.astype(int)
out['Year'] = out['Date'].dt.year
out['Market'] = 'Pettah/Peliyagoda Colombo'
features = ['Crop', 'Market', 'Year', 'Month', 'WeekOfYear', 'CurrentPrice_RsKg', 'Lag1Price_RsKg', 'Lag2Price_RsKg', 'Lag3Price_RsKg', 'Rolling7Mean_RsKg', 'Rolling28Mean_RsKg', 'Rolling7Std_RsKg']
out = out.dropna(subset=features + ['NextWeekWholesalePrice_RsKg'])
out = out[['Date', 'NextWeekStart'] + features + ['NextWeekWholesalePrice_RsKg']]
out.to_csv(ROOT / 'data' / 'preprocessed' / 'colombo_weekly_prices_preprocessed.csv', index=False)
print('Daily crop-date records:', len(daily))
print('Final next-week price records:', len(out))
print('Saved colombo_weekly_prices_preprocessed.csv')

The target is the following calendar week mean. Lag and rolling columns use only prices available before that target week.

In [ ]:
import subprocess

print(subprocess.run(["git", "status", "--short"], capture_output=True, text=True).stdout)
subprocess.run(["git", "add", "notebooks/03_price_data_jayakody.ipynb"], check=True)
subprocess.run(["git", "add", "data/preprocessed/colombo_weekly_prices_preprocessed.csv"], check=True)
subprocess.run(["git", "commit", "-m", "Prepare Colombo next-week price data"], check=False)
subprocess.run(["git", "push", "origin", "main"], check=True)
print("Contribution pushed to main.")